---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: d:\ADC 1\AI engineering\echochamber-project-team3
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [4]:
import pandas as pd
import random


corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[NicusorDanRO] Baliverne. Justiția e pe bani indiferent ca vb de Germania, Franța, Anglia, SUA.
[georgesimionoficial] Așa da, domnule Simion! Tot înainte, până la victoria finală!❤️💛💙🙏🙏🙏
[@CălinGeorgescu-CanalulOficial] La multi multi ani dl Presedinte. Va iubim. Dumnezeu sa va ocroteasca pe dvs si 


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [6]:
# modifica dupa preferinte

AXA_1 = "media_distrust"
AXA_2 = "religious_frame"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [ ]:
AXA_1_DEFINITION = """
media_distrust măsoară dacă textul exprimă neîncredere în presă, jurnaliști,
televiziuni sau media mainstream.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
religious_frame măsoară dacă textul folosește limbaj religios pentru a interpreta politica.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [8]:
MINI_PROMPT = f"""
Ești un expert în analiză de text și adnotare politică.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un expert în analiză de text și adnotare politică.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. media_distrust
2. religious_frame
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
media_distrust = 0 / 1 / 2
religious_frame = 0 / 1 / 2
DEFINIȚII:

media_distrust măsoară dacă textul exprimă neîncredere în presă, jurnaliști,
televiziuni sau media mainstream.
0 = absent
1 = prezent
2 = dominant


religious_frame măsoară dacă textul folosește limbaj religios pentru a interpreta politica.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = pre

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [9]:
TESTS = corpus.sample(5, random_state=32)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
197,yt_3MhSvAqrFOA_Ugwnr5YgsmibiIJNaGl4AaABAg,turcescu111,EXCLUSIVITATE: &quot;Speram să fie informat Si...,"I-ați terminat pe cei din ,,sistem"" prin preze..."
182,yt_GszmQ-M_vNY_UgwdA0B9Cc-rPNGtPC94AaABAg,TVRcanaluloficial,Nadia Comăneci a fost omagiată în plenul Parla...,Praise God 🙏 Nadia was then and we love her bu...
419,yt_VDiv4TBODF8_UgxWEdA734-0cxqELad4AaABAg,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Infectia asta a fost aleasa primar de catre oa...
54,yt_h5-18BWyFJ4_Ugz9XjOFTXTr76U3Et94AaABAg,georgesimionoficial,Dumnezeu e mare! Explicăm situația din România...,Ne-ar fi prins și noua bine sustinerea america...
170,yt_97qXZe3_J_k_UgyFOXq3-6IBVqwnhmB4AaABAg,turcescu111,Georgescu le-a dat la operație!,"NU, TURUL DOI DL.GEORGESCU ESTE ADEVARATUL PRE..."


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [10]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [11]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [12]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
I-ați terminat pe cei din ,,sistem" prin prezentarea celor scrise în pag.8;9 din cartea ÎNAPOI LA TURUL DOI . FELICITĂRI !

OUTPUT MODEL:
```json
{
  "target": "sistemul",
  "stance": "pro",
  "tone": "acuzator",
  "media_distrust": 0,
  "religious_frame": 0
}
```
COMENTARIU:
Praise God 🙏 Nadia was then and we love her but too much propaganda away too much !!!

OUTPUT MODEL:
```json
{
  "target": "Nadia Comăneci",
  "stance": "pro",
  "tone": "afectiv",
  "media_distrust": 0,
  "religious_frame": 1
}
```
COMENTARIU:
Infectia asta a fost aleasa primar de catre oameni - asta este de fapt adevarata tragedie a Romaniei.

OUTPUT MODEL:
```json
{
  "target": "Negoiță",
  "stance": "anti",
  "tone": "acuzator",
  "media_distrust": 0,
  "religious_frame": 0
}
```
COMENTARIU:
Ne-ar fi prins și noua bine sustinerea americanilor, dar ne-au abandonat și au ales doar Polonia

OUTPUT MODEL:
```json
{
  "target": "SUA",
  "stance": "anti",
  "tone": "acuzator",
  "media_distrust": 0,
  "r

In [ ]:
## Pasul 6 — Interpretare scurtă
Completează în notebook, în 3–5 rânduri:
- Ce două axe ai ales?
Le-am lăsat pe cele inițiale
- De ce le-ai ales?
Mi s-au parut interesante
- Modelul a returnat JSON corect?
Da
- Care a fost cea mai mare problemă?
Nu au fost probleme
- Ce ai schimba în prompt?
I-aș arăta modelului exact un exemplu de comentariu pentru nota 1 și un exemplu pentru nota 2, ca să înțeleagă mai bine diferența dintre ele și să nu mai exagereze scorurile